In [10]:
import pandas as pd
import numpy as np
import json
import re, pytz, os, requests, sys
from pathlib import Path
from datetime import datetime
import sys
sys.path.append("/workspaces/service-data")

from src.clean import clean_percentage, clean_fiscal_yr, normalize_string, standardize_column_names
from src.load import load_csv
from src.export import export_to_csv
from src.merge import merge_si, merge_ss
from src.utils import dept_list, program_list
from main import get_config

import pandas as pd
import numpy as np
import pytz
from pathlib import Path

base_dir = Path.cwd()
parent_dir = base_dir.parent
config = get_config()
# dept = dept_list(config)

# si_url = https://github.com/gcperformance/service-data/releases/latest/download/si.csv
# si = pd.read_csv(si_url, keep_default_na=False, na_values='', delimiter=';')

# si_path = parent_dir / 'outputs' / 'si.csv'
# si = pd.read_csv(si_path, keep_default_na=False, na_values='', delimiter=';', engine='python', skipfooter=2)

# sid_registry = load_csv(sid_registry_path, config)

In [ ]:
org_var = load_csv('org_var.csv', config)

frames_en = []
frames_fr = []
frames_enfr = []
df = pd.DataFrame

for fiscal_yr, url in config['program_csv_urls_en'].items():
    filename = url.split('/')[-1].split('.')[0]+'.csv'
    
    df = load_csv(filename, config)
    df['filename'] = filename
    df['fiscal_yr'] = fiscal_yr
    frames_en += [df]

for fiscal_yr, url in config['program_csv_urls_fr'].items():
    filename = url.split('/')[-1].split('.')[0]+'.csv'

    df = load_csv(filename, config)
    df['filename'] = filename
    df['fiscal_yr'] = fiscal_yr
    frames_fr += [df]

for fiscal_yr, url in config['program_csv_urls_enfr'].items():
    filename = url.split('/')[-1].split('.')[0]+'.csv'

    df = load_csv(filename, config)
    df['filename'] = filename
    df['fiscal_yr'] = fiscal_yr
    frames_enfr += [df]

# --- Process English Program Data ---
program_df_en = pd.concat(frames_en, ignore_index=True)

# Determine the org_id using the org name variants
program_df_en = program_df_en.merge(
    org_var,
    how='left',
    left_on='Entity_Entite_eng', #This name is not always constant, changed in 2026 -GE
    right_on='org_name_variant'
)

# Resolve program codes and names, taking the core responsibility if the program is null
program_df_en['program_id'] = program_df_en[
    'Prog-inv-code_Code-rep-prog'
].combine_first(
    program_df_en['Prog-core-resp-code_Code-prog-resp-essent']
)

program_df_en['program_en'] = program_df_en[
    'Prog-inv-name_Nom-rep-prog_eng'
].combine_first(
    program_df_en['Prog-core-resp-name_Nom-prog-resp-essent_eng']
)

# Set index for merging
program_df_en.set_index(['fiscal_yr', 'org_id', 'program_id'], inplace=True)

# --- Process French Program Data ---
program_df_fr = pd.concat(frames_fr, ignore_index=True)

# Determine the org_id using the org name variants
program_df_fr = program_df_fr.merge(
    org_var,
    how='left',
    left_on='Entity_Entite_fra', #This name is not always constant - beware.
    right_on='org_name_variant'
)

# Resolve program codes and names, taking the core responsibility if the program is null
program_df_fr['program_id'] = program_df_fr[
    'Prog-inv-code_Code-rep-prog'
].combine_first(
    program_df_fr['Prog-core-resp-code_Code-prog-resp-essent']
)

program_df_fr['program_fr'] = program_df_fr[
    'Prog-inv-name_Nom-rep-prog_fra'
].combine_first(
    program_df_fr['Prog-core-resp-name_Nom-prog-resp-essent_fra']
)

program_df_fr.set_index(['fiscal_yr', 'org_id', 'program_id'], inplace=True)

# --- Process Bilingual Program Data ---
program_df_enfr = pd.concat(frames_enfr, ignore_index=True)

# Determine the org_id using the org name variants
program_df_enfr = program_df_enfr.merge(
    org_var,
    how='left',
    left_on='Entity_Entite_eng',
    right_on='org_name_variant'
)

# Resolve program codes and names, taking the core responsibility if the program is null
program_df_enfr['program_id'] = program_df_enfr[
    'Prog-inv-code_Code-rep-prog'
].combine_first(
    program_df_enfr['Prog-core-resp-code_Code-prog-resp-essent']
)

program_df_enfr['program_en'] = program_df_enfr[
    'Prog-inv-name_Nom-rep-prog_eng'
].combine_first(
    program_df_enfr['Prog-core-resp-name_Nom-prog-resp-essent_eng']
)

program_df_enfr['program_fr'] = program_df_enfr[
    'Prog-inv-name_Nom-rep-prog_fra'
].combine_first(
    program_df_enfr['Prog-core-resp-name_Nom-prog-resp-essent_fra']
)

program_df_enfr.set_index(['fiscal_yr', 'org_id', 'program_id'], inplace=True)

# --- Merge English and French Data ---
program_df = pd.merge(
    program_df_en,
    program_df_fr,
    how='outer',
    left_index=True,
    right_index=True,
    indicator=True
)

# --- Concat bilingual data ---
program_df = pd.concat([program_df, program_df_enfr])    

# Keep only necessary columns, noting that the index has org id, fiscal yr, and code
program_df = program_df[['program_en', 'program_fr']].reset_index()

# --- Select Latest Fiscal Year per org_id–program_id ---
program_df = program_df.sort_values('fiscal_yr')

latest_idx = program_df.groupby(['org_id', 'program_id'])['fiscal_yr'].idxmax()
program_df = program_df.loc[latest_idx, ['org_id', 'program_id', 'fiscal_yr', 'program_en', 'program_fr']]

program_df['org_id'] = pd.to_numeric(program_df['org_id'], errors = 'coerce').fillna(0).astype('Int64')

program_df.rename(columns={'fiscal_yr':'latest_valid_fy'}, inplace=True)

UTILS_DIR = config['output_dir'] / config['utils_dir']
export_to_csv(
    data_dict={'program_list':program_df},
    output_dir=UTILS_DIR
    )
